# Toktogul Battery - water surface of Kyrgyzstan's main reservoir measured by NISAR radar

**Team Paparizi - NASA Space Apps Challenge 2026 - challenge "Dancing with the SARs"**

What this notebook does:
1. logs in to NASA Earthdata (you type your own login and password - they stay inside this notebook);
2. finds every NISAR radar scene over the Toktogul reservoir taken from one orbit;
3. reads **only the reservoir window** straight from NASA's cloud (a full scene is about 2 GB - we never download it);
4. separates water from dry ground using the whole time series (see step 6 for why one image is not enough);
5. saves the pictures and the table used by the website.

**How to run:** menu *Runtime -> Run all*. When the login cell asks, type your Earthdata username and password.

Data: NISAR L2 GCOV (L-band radar backscatter, HH and HV, 20 m pixels), NASA/ISRO, distributed by the Alaska Satellite Facility DAAC.

*AI disclosure: the code in this notebook was written with the help of an AI assistant (Claude by Anthropic). The team ran it and checked the results against the images.*

## 0. Install the tools

In [ ]:
!pip -q install earthaccess h5py pyproj scipy pillow matplotlib

## 1. Settings

In [ ]:
# ---------- SETTINGS (you can change these) ----------
AOI = dict(lon_min=72.55, lon_max=73.45, lat_min=41.58, lat_max=42.00)   # box around the Toktogul reservoir
SEED = (72.874, 41.806)            # lon, lat of a point that is ALWAYS water (middle of the main basin)
MAIN_TRACK = ("092", "D", "067")   # (track, direction, frame) - this frame covers the whole reservoir
SHORT_NAMES = ["NISAR_L2_GCOV_PROVISIONAL_V1", "NISAR_L2_GCOV_V1"]  # NASA collections to search
FREQ = "A"                         # frequency A = 20 m pixels (B = 80 m)
OUT = "out"                        # output folder
BLOCK = 8 * 1024 * 1024            # network read block (bytes)
SMOOTH = 3                         # speckle smoothing window (pixels)

# Reservoir reference numbers (billion m3 / km2) - public design figures
V_FULL, V_DEAD, A_FULL = 19.5, 5.5, 284.3
# Official volumes announced by the Ministry of Energy (date, billion m3, source)
OFFICIAL = [
    ("2026-07-08", 10.2,   "https://economist.kg/enierghietika/2026/07/08/toktogul-voda-11-5-mlrd/"),
    ("2026-10-01", 11.4,   "https://economist.kg/enierghietika/2026/10/19/vyrabotka-980-mln-2026/"),
    ("2026-10-15", 11.345, "https://economist.kg/enierghietika/2026/10/19/vyrabotka-980-mln-2026/"),
]
# Reference figures used on the website (billion m3)
V_TARGET = 11.5


## 2. Functions for reading the data

In [ ]:
import os, re, json, time, zipfile, datetime as dt
import numpy as np
import h5py
from pyproj import Transformer
from scipy import ndimage as ndi
from PIL import Image
import matplotlib.pyplot as plt

GRID = "/science/LSAR/GCOV/grids/frequency{}"
NAME_RE = re.compile(
    r"NISAR_L2_[A-Z]{2}_GCOV_(?P<cycle>\d{3})_(?P<track>\d{3})_(?P<dir>[AD])_(?P<frame>\d{3})_"
    r"(?P<mode>\w{4})_(?P<pol>\w{4})_(?P<src>\w)_(?P<start>\d{8}T\d{6})_(?P<end>\d{8}T\d{6})_(?P<crid>\w{6})_")


def parse_name(name):
    m = NAME_RE.search(name)
    return m.groupdict() if m else None


# ---------- 1. find the scenes ----------
def find_scenes(tracks):
    """Ask NASA's catalogue for every GCOV scene over the reservoir on the given orbits."""
    import earthaccess
    bbox = (AOI["lon_min"], AOI["lat_min"], AOI["lon_max"], AOI["lat_max"])
    found = {}
    for sn in SHORT_NAMES:
        try:
            res = earthaccess.search_data(short_name=sn, bounding_box=bbox, count=-1)
        except Exception as e:
            print(f"  search in {sn} failed: {e}")
            continue
        print(f"  {sn}: {len(res)} products touch the area")
        for g in res:
            for url in g.data_links(access="external"):
                if not url.endswith(".h5") or url.endswith("_QA_STATS.h5"):
                    continue
                info = parse_name(url.rsplit("/", 1)[-1])
                if not info or (info["track"], info["dir"], info["frame"]) not in tracks:
                    continue
                d = info["start"][:8]
                s = dict(url=url, date=f"{d[:4]}-{d[4:6]}-{d[6:]}", time_utc=info["start"],
                         track=info["track"], dir=info["dir"], frame=info["frame"],
                         mode=info["mode"], pol=info["pol"], crid=info["crid"], collection=sn)
                k = (s["track"], s["dir"], s["frame"], s["date"])
                if k not in found or s["crid"] > found[k]["crid"]:
                    found[k] = s
    return sorted(found.values(), key=lambda s: (s["track"], s["date"]))


# ---------- 2. read only our window from the remote file ----------
def open_remote(url):
    """Open a NISAR file that stays on NASA's cloud; only the parts we ask for travel over the network."""
    import earthaccess
    fs = earthaccess.get_fsspec_https_session()
    try:
        fo = fs.open(url, cache_type="background", block_size=BLOCK)
    except Exception:
        fo = fs.open(url, block_size=BLOCK)
    return h5py.File(fo, "r")


OPENER = open_remote


def check_access(url):
    """True if we can really read the file (first bytes must be the HDF5 signature)."""
    import earthaccess
    fs = earthaccess.get_fsspec_https_session()
    with fs.open(url, block_size=1024 * 1024) as f:
        head = f.read(8)
    return head == b"\x89HDF\r\n\x1a\n", head


def aoi_xy(epsg):
    tr = Transformer.from_crs(4326, epsg, always_xy=True)
    lo = np.linspace(AOI["lon_min"], AOI["lon_max"], 40)
    la = np.linspace(AOI["lat_min"], AOI["lat_max"], 40)
    lons = np.r_[lo, lo, np.full(40, AOI["lon_min"]), np.full(40, AOI["lon_max"])]
    lats = np.r_[np.full(40, AOI["lat_min"]), np.full(40, AOI["lat_max"]), la, la]
    xs, ys = tr.transform(lons, lats)
    return xs.min(), xs.max(), ys.min(), ys.max()


def read_window(url):
    with OPENER(url) as h5:
        g = h5[GRID.format(FREQ)]
        x = np.asarray(g["xCoordinates"][:], "float64")
        y = np.asarray(g["yCoordinates"][:], "float64")
        pr = g["projection"]
        try:
            epsg = int(np.asarray(pr[()]).ravel()[0])
        except Exception:
            epsg = 0
        if not (1024 < epsg < 100000):
            epsg = int(pr.attrs.get("epsg_code", 0))
        xmin, xmax, ymin, ymax = aoi_xy(epsg)
        ix = np.where((x >= xmin) & (x <= xmax))[0]
        iy = np.where((y >= ymin) & (y <= ymax))[0]
        if ix.size < 10 or iy.size < 10:
            raise ValueError("the area of interest is outside this scene")
        c0, c1, r0, r1 = int(ix.min()), int(ix.max()) + 1, int(iy.min()), int(iy.max()) + 1
        out = dict(x=x[c0:c1], y=y[r0:r1], epsg=epsg)
        for p in ("HHHH", "HVHV"):
            if p in g:
                out[p] = np.asarray(g[p][r0:r1, c0:c1], "float32")
        if "HHHH" not in out:
            raise ValueError(f"no HH layer in this file; layers present: {list(g.keys())}")
    return out


def load_scene(s):
    """Read the window once and keep it on disk, so re-running the notebook is fast."""
    os.makedirs(f"{OUT}/cache", exist_ok=True)
    f = f"{OUT}/cache/{s['track']}{s['dir']}{s['frame']}_{s['date']}.npz"
    if os.path.exists(f):
        z = np.load(f)
        return {k: (int(z[k]) if k == "epsg" else z[k]) for k in z.files}
    t0 = time.time()
    d = read_window(s["url"])
    np.savez(f, **d)
    nod = 100 * float(np.mean(~np.isfinite(d["HHHH"])))
    print(f"    {s['date']}: read {d['HHHH'].shape[1]} x {d['HHHH'].shape[0]} px in {time.time() - t0:.0f} s, no-data {nod:.1f}%")
    return d


def seed_index(d):
    tr = Transformer.from_crs(4326, d["epsg"], always_xy=True)
    sx, sy = tr.transform(*SEED)
    return int(np.abs(d["y"] - sy).argmin()), int(np.abs(d["x"] - sx).argmin())


def pixel_km2(d):
    return abs(d["x"][1] - d["x"][0]) * abs(d["y"][1] - d["y"][0]) / 1e6


# ---------- 3. from radar power to decibels ----------
def smooth_power(p, size):
    """Average neighbouring pixels (in power, not dB) to calm the radar 'grain' (speckle). Ignores no-data."""
    valid = np.isfinite(p) & (p > 0)
    num = ndi.uniform_filter(np.where(valid, p, 0).astype("float32"), size=size, mode="nearest")
    den = ndi.uniform_filter(valid.astype("float32"), size=size, mode="nearest")
    out = np.full(p.shape, np.nan, "float32")
    ok = den > 0.5
    out[ok] = num[ok] / den[ok]
    return out


def to_db(p):
    with np.errstate(divide="ignore", invalid="ignore"):
        return (10 * np.log10(p)).astype("float32")


def channels(d):
    """dB images: HH, HV and their average (MIX)."""
    ch = {"HH": to_db(smooth_power(d["HHHH"], SMOOTH))}
    if "HVHV" in d:
        ch["HV"] = to_db(smooth_power(d["HVHV"], SMOOTH))
        ch["MIX"] = (ch["HH"] + ch["HV"]) / 2
    return ch


# ---------- 4. pictures ----------
def downsample(a, k):
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    b = a[:h, :w].reshape(h // k, k, w // k, k)
    with np.errstate(invalid="ignore"):
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return np.nanmean(b, axis=(1, 3))


def to_gray(db, vmin, vmax):
    g = np.clip((db - vmin) / (vmax - vmin), 0, 1)
    return (np.nan_to_num(g, nan=0) * 255).astype("uint8")


def north_up(a, d):
    """Rows must go from north to south for a picture."""
    return a if d["y"][0] > d["y"][-1] else a[::-1]


def corners_lonlat(d):
    """lon/lat of the 4 image corners (top-left, top-right, bottom-right, bottom-left) for a web map."""
    tr = Transformer.from_crs(d["epsg"], 4326, always_xy=True)
    dx = abs(d["x"][1] - d["x"][0]) / 2
    dy = abs(d["y"][1] - d["y"][0]) / 2
    x0, x1 = d["x"].min() - dx, d["x"].max() + dx
    y0, y1 = d["y"].min() - dy, d["y"].max() + dy
    return [list(map(float, tr.transform(x, y))) for x, y in ((x0, y1), (x1, y1), (x1, y0), (x0, y0))]


def days(datestr):
    return dt.date.fromisoformat(datestr).toordinal()


## 3. Log in to NASA Earthdata
A free account is needed: https://urs.earthdata.nasa.gov/users/new

In [ ]:
import earthaccess
auth = earthaccess.login()   # asks for your Earthdata username and password; they stay inside this notebook
print("Logged in to NASA Earthdata:", auth.authenticated)


## 4. Find the scenes and check that we can read them
We only compare scenes from **the same orbit** (same track and frame): the radar then looks at the mountains from exactly the same side every time.

In [ ]:
tracks = [MAIN_TRACK]
print("Searching NASA's catalogue...")
SCENES = find_scenes(tracks)
print(f"\n{len(SCENES)} scenes found on the chosen orbit(s):")
for s in SCENES:
    print(f"  {s['date']}  track {s['track']}{s['dir']}  frame {s['frame']}  mode {s['mode']} {s['pol']}  {s['crid']}  {s['collection']}")
assert SCENES, "No scenes found. Check MAIN_TRACK and AOI in the settings, and that the Earthdata login worked."

ok, head = check_access(SCENES[0]["url"])
print("\nCan we read the data? ->", "YES" if ok else "NO")
if not ok:
    print("NASA did not let us read the file. Fix: open the link below in your browser while you are logged in to")
    print("Earthdata, accept the terms if it asks, then run this cell again.\n", SCENES[0]["url"])


## 5. Read every date and put all dates on one 80 m grid
Neighbouring pixels are averaged to calm the radar grain (speckle); brightness is in decibels (dB).

In [ ]:
# Read the reservoir window on every date (a few minutes the first time) and bring all dates to one 80 m grid.
ANALYSIS_STEP = 4          # 4 x 20 m = 80 m: averaging calms the radar noise
main = [s for s in SCENES if (s["track"], s["dir"], s["frame"]) == MAIN_TRACK]
assert len(main) >= 3, "Fewer than 3 scenes on the main orbit - the method needs a time series."
HH80, HV80, DATES, TIMES, GEO = [], [], [], [], None
for s in main:
    try:
        d = load_scene(s)
    except Exception as e:
        print(f"    {s['date']}: SKIPPED ({e})")
        continue
    if "HVHV" not in d or np.mean(~np.isfinite(d["HHHH"])) > 0.3:
        print(f"    {s['date']}: SKIPPED (no HV layer, or more than 30% of the window has no data)")
        continue
    ch = channels(d)
    hh = downsample(north_up(ch["HH"], d), ANALYSIS_STEP).astype("float16")     # float16 keeps the stack small
    hv = downsample(north_up(ch["HV"], d), ANALYSIS_STEP).astype("float16")
    if GEO is None:
        r, c = seed_index(d)
        if d["y"][0] < d["y"][-1]:
            r = len(d["y"]) - 1 - r
        GEO = dict(seed=(r // ANALYSIS_STEP, c // ANALYSIS_STEP), px_km2=pixel_km2(d) * ANALYSIS_STEP ** 2,
                   corners=corners_lonlat(d), epsg=int(d["epsg"]), shape=hh.shape)
    if hh.shape != GEO["shape"]:
        print(f"    {s['date']}: SKIPPED (this scene sits on a different grid)")
        continue
    HH80.append(hh); HV80.append(hv); DATES.append(s["date"]); TIMES.append(s["time_utc"])
HH80, HV80 = np.stack(HH80), np.stack(HV80)
print(f"{len(DATES)} dates on a {HH80.shape[2]} x {HH80.shape[1]} grid, pixel {GEO['px_km2'] ** 0.5 * 1000:.0f} m")

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
for a, i in zip(ax, (0, len(DATES) - 1)):
    a.imshow(HH80[i].astype("float32"), cmap="gray", vmin=-23, vmax=-2)
    a.plot(GEO["seed"][1], GEO["seed"][0], "r+", ms=14)
    a.set_title(f"{DATES[i]}  (HH; red cross = seed point, must be on water)"); a.axis("off")
plt.tight_layout(); plt.show()


## 6. Water on every date

**Why a simple threshold fails here.** NISAR's L-band radar sees calm water as very dark - but dry, smooth ground is just as dark. On a dry day a threshold counts the dry plain around the reservoir as water (our first attempt gave 700 km2 instead of about 215).

**What works.** After rain the ground turns bright and the water does not change. So:
1. the water that is dark on *every* date gives the brightness of water on each date;
2. *rain-check dates* are the dates when the land next to the reservoir is clearly brighter than usual (wet ground, most likely after rain);
3. the water body = what stays dark on the rain-check dates (bays cut off by a narrow neck are re-attached);
4. for any other date a pixel counts as water only if it looks the way it does on the rain-check dates - on that date and on every date between it and the rain-check date. A one-date exception (wind, radar stripes) is allowed;
5. the test is repeated with a stricter and a looser setting: that is the error band.

**Check.** `check_method.py` in this repository runs the same cells on simulated scenes with a known answer (dry plain as dark as water, two wet dates, stripes, wind): the measured area stays within 1 km2 of the truth. Real ground is messier.

**Limits.** The analysis grid is 80 m, so the total area is uncertain by several km2. Changes are only followed inside the rain-check water body; growth between rain-check dates on dry days can be undercounted. The narrow canyon next to the dam is only partly included.

In [ ]:
# ---------- WATER FROM THE WHOLE TIME SERIES (method v2) ----------
# Why not a simple threshold on one image? At L-band, calm water and dry smooth ground are both very dark,
# so one image cannot tell them apart. But after rain the ground gets bright and the water does not.
# So we (1) find the "rain-check" dates, (2) take as water only what stays dark on them,
# (3) follow every shoreline pixel back and forth in time: it is water on a date only if it looks the same
#     as it does on the rain-check dates, when we know it is water.
ANALYSIS_STEP = 4          # analysis grid = 4 x 20 m = 80 m (averaging calms the radar noise)
RAIN_BOOST_DB = 1.5        # a date is a rain-check date if the land is this much brighter than usual
CORE_MARGIN_DB = 2.5       # "always dark": HV never rises more than this above the water level
EXTENT_MARGINS = (1.3, 4.0)  # on rain-check dates: HV / HH must stay within these margins of the water level (dB)
BAY_MIN_KM2, BAY_LINK_KM = 0.5, 0.8   # bays cut off by a narrow neck are re-attached to the main body
QUANTILES = (99.0, 99.5, 99.9)        # strict / main / loose setting of the per-date test (gives the error band)


def fill_gaps(a):
    bad = ~np.isfinite(a)
    if bad.any():
        idx = ndi.distance_transform_edt(bad, return_distances=False, return_indices=True)
        a = a[tuple(idx)]
    return a


def fill_small_holes(mask, max_px):
    holes = ndi.binary_fill_holes(mask) & ~mask
    lab, _ = ndi.label(holes)
    sizes = np.bincount(lab.ravel())
    sizes[0] = max_px + 1
    return mask | (sizes <= max_px)[lab]


def analyse_stack(HH, HV, dates, seed_rc, px_km2):
    """HH, HV: arrays (dates, rows, cols) in dB on one common north-up grid. Returns masks and a table."""
    n = len(dates)
    if n < 3:
        raise RuntimeError("the method needs at least 3 dates from the same orbit")
    HH = np.stack([fill_gaps(x.astype("float32")) for x in HH])
    HV = np.stack([fill_gaps(x.astype("float32")) for x in HV])
    r0, c0 = seed_rc
    px_m = (px_km2 ** 0.5) * 1000

    def body(mask, ref):                 # the connected patch of `mask` that shares the most pixels with `ref`
        lab, _ = ndi.label(mask)
        hits = np.bincount(lab[ref & mask], minlength=2)
        hits[0] = 0
        return (lab == hits.argmax()) if hits.max() else np.zeros(mask.shape, bool)

    # 1. water that is dark on every date -> the reference brightness of water on each date
    rr, cc = np.ogrid[:HH.shape[1], :HH.shape[2]]
    around_seed = (rr - r0) ** 2 + (cc - c0) ** 2 <= (1500 / px_m) ** 2          # 1.5 km around the seed point
    seed_v = np.array([np.median(HV[t][around_seed]) for t in range(n)])
    core = body((HV - seed_v[:, None, None]).max(0) < CORE_MARGIN_DB, around_seed)
    if core.sum() * px_km2 < 5:
        raise RuntimeError("could not find the permanent water around the seed point - check SEED")
    ref_v = np.array([np.median(HV[t][core]) for t in range(n)])
    ref_h = np.array([np.median(HH[t][core]) for t in range(n)])
    dV, dH = HV - ref_v[:, None, None], HH - ref_h[:, None, None]

    # 2. rain-check dates: the land next to the reservoir is clearly brighter than usual
    ring = (ndi.distance_transform_edt(~core) * px_m < 3200) & ~core
    land_h = np.array([np.median(HH[t][ring]) for t in range(n)])
    anchors = [t for t in range(n) if land_h[t] >= np.median(land_h) + RAIN_BOOST_DB]
    rain_found = bool(anchors)
    if not anchors:                      # no clearly wet date: fall back to the date with the brightest land
        anchors = [int(np.argmax(land_h))]

    # 3. the water body on the rain-check dates (+ bays that a narrow neck cuts off at this resolution)
    bV, bH = dV[anchors].mean(0), dH[anchors].mean(0)
    dark_rain = (bV < EXTENT_MARGINS[0]) & (bH < EXTENT_MARGINS[1])
    main = body(dark_rain, core)
    lab, _ = ndi.label(dark_rain)
    sizes = np.bincount(lab.ravel())
    near = ndi.distance_transform_edt(~main) * px_m <= BAY_LINK_KM * 1000
    M = main.copy()
    for l in np.unique(lab[near & (lab > 0)]):
        if sizes[l] * px_km2 >= BAY_MIN_KM2:
            M |= lab == l
    # the radar leaves thin bright stripes across very dark water; close gaps up to 2 pixels wide (never over bright land)
    k3 = np.ones((3, 3), bool)
    M = M | (ndi.binary_closing(M, structure=k3, iterations=1) & (bH < 7.0))
    M = fill_small_holes(M, int(0.3 / px_km2))            # specks inside the water (anything bigger stays: it may be an island)

    # 4. every pixel against its own "water look"; the margins come from deep water, which is water for sure
    eV, eH = dV - bV, dH - bH
    deep = ndi.binary_erosion(core & main, iterations=max(int(round(500 / px_m)), 1))   # always-dark water, > 500 m from any shore
    if deep.sum() < 500:
        deep = core & main
    if deep.sum() < 50:
        raise RuntimeError("the permanent water is too small to calibrate the test - check SEED and the list of dates")
    first, last = min(anchors), max(anchors)

    def masks_for(q):
        mv, mh = np.percentile(eV[:, deep], q), np.percentile(eH[:, deep], q)
        dark = (eV < mv) & (eH < mh)
        ok = dark.copy()
        for t in range(1, n - 1):                       # a one-date dropout (wind, stripes) is not land
            ok[t] |= dark[t - 1] & dark[t + 1]
        out = [None] * n
        alive = M.copy()
        tidy = lambda w: fill_small_holes(ndi.binary_closing(w, structure=k3, iterations=1) & M, int(0.3 / px_km2))
        for t in range(first, -1, -1):                  # backwards from the first rain-check date
            alive = alive & ok[t]
            out[t] = tidy(alive)
        for t in range(first, last + 1):
            out[t] = M.copy()
        alive = M.copy()
        for t in range(last + 1, n):                    # forwards from the last rain-check date
            alive = alive & ok[t]
            out[t] = tidy(alive)
        return out, float(mv), float(mh)

    strict, main_m, loose = (masks_for(q) for q in QUANTILES)
    rows = []
    for t in range(n):
        a_s, a_m, a_l = (m[0][t].sum() * px_km2 for m in (strict, main_m, loose))
        rows.append(dict(date=dates[t], area_km2=round(float(a_m), 2), area_lo=round(float(min(a_s, a_l)), 2),
                         area_hi=round(float(max(a_s, a_l)), 2), rain_check=bool(t in anchors),
                         water_hv_db=round(float(ref_v[t]), 2), water_hh_db=round(float(ref_h[t]), 2),
                         land_hh_db=round(float(land_h[t]), 2)))
    # deep water wrongly called land on the first date = how much of the "growth" is noise
    noise = float((deep & ~main_m[0][0]).sum() * px_km2) * (M.sum() / max(deep.sum(), 1))   # scaled to the whole water body
    return dict(rows=rows, masks=main_m[0], extent=M, core=core, anchors=[dates[t] for t in anchors], rain_found=rain_found,
                margins_db=dict(hv=round(main_m[1], 2), hh=round(main_m[2], 2)),
                noise_km2=round(noise, 2), bays_km2=round(float((M & ~main).sum() * px_km2), 2))


In [ ]:
R = analyse_stack(HH80, HV80, DATES, GEO["seed"], GEO["px_km2"])
rows = R["rows"]
print("Rain-check dates:", ", ".join(R["anchors"]))
if not R["rain_found"]:
    print("WARNING: no clearly rainy date in this series. The date with the brightest land was used instead; "
          "dry ground next to the reservoir may be counted as water.")
print(f"Water body on the rain-check dates: {R['extent'].sum() * GEO['px_km2']:.1f} km2 (of which re-attached bays {R['bays_km2']} km2)")
print(f"Margins of the per-date test: HV +{R['margins_db']['hv']} dB, HH +{R['margins_db']['hh']} dB; noise in the growth estimate about {R['noise_km2']} km2\n")
for r in rows:
    print(f"  {r['date']}  water {r['area_km2']:7.1f} km2   (stricter / looser test: {r['area_lo']:.1f} .. {r['area_hi']:.1f}){'   <- rain-check date' if r['rain_check'] else ''}")
print(f"\nChange from {rows[0]['date']} to the highest date: {max(r['area_km2'] for r in rows) - rows[0]['area_km2']:+.1f} km2")

t = [dt.date.fromisoformat(r["date"]) for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(18, 5.2), gridspec_kw=dict(width_ratios=[1, 1.25]))
ax[0].fill_between(t, [r["area_lo"] for r in rows], [r["area_hi"] for r in rows], alpha=0.25, label="stricter / looser test")
ax[0].plot(t, [r["area_km2"] for r in rows], "o-", label="water surface from NISAR")
ax[0].plot([x for x, r in zip(t, rows) if r["rain_check"]], [r["area_km2"] for r in rows if r["rain_check"]], "o", mfc="white", ms=11, label="rain-check dates")
for d, v, _ in OFFICIAL:
    ax[0].axvline(dt.date.fromisoformat(d), ls=":", color="k"); ax[0].text(dt.date.fromisoformat(d), ax[0].get_ylim()[0], f" official {v}", rotation=90, va="bottom", fontsize=9)
ax[0].set_ylabel("km2"); ax[0].set_title("Toktogul reservoir: water surface"); ax[0].legend(); ax[0].grid(alpha=0.3)
i1 = DATES.index(R["anchors"][0])
ax[1].imshow(HH80[i1].astype("float32"), cmap="gray", vmin=-23, vmax=-2)
ax[1].contour(R["masks"][i1], levels=[0.5], colors="cyan", linewidths=0.7)
gain = np.zeros(R["masks"][0].shape + (4,)); gain[R["masks"][i1] & ~R["masks"][0]] = (1, 0.72, 0.1, 0.85); ax[1].imshow(gain)
ax[1].set_title(f"{DATES[i1]} (after rain): cyan = shoreline, amber = water gained since {DATES[0]}"); ax[1].axis("off")
plt.tight_layout()
os.makedirs(f"{OUT}/web", exist_ok=True)
plt.savefig(f"{OUT}/web/summary.png", dpi=110)
plt.show()


## 7. Save everything for the website
Writes `site_data.json`, `series.csv` and `summary.png` into `toktogul_out.zip`. Put `template.html` next to this notebook to get the finished `index.html` as well.

In [ ]:
# Save the data for the website. If template.html is in the same folder, the finished index.html is written too.
import base64, io

def b64(img, fmt, **kw):
    b = io.BytesIO(); img.save(b, fmt, **kw); return base64.b64encode(b.getvalue()).decode()

images, masks = [], []
for i in range(len(DATES)):
    g = np.clip((HH80[i].astype("float32") + 23.0) / 21.0, 0, 1)
    images.append(b64(Image.fromarray((np.nan_to_num(g, nan=0) ** 1.15 * 255).astype("uint8")), "JPEG", quality=82, optimize=True))
    masks.append(b64(Image.fromarray((R["masks"][i] * 255).astype("uint8"), "L").convert("1"), "PNG", optimize=True))
site = dict(dates=DATES, times_utc=TIMES, rows=rows, size=[int(HH80.shape[2]), int(HH80.shape[1])], corners=GEO["corners"],
            orbit=dict(track=MAIN_TRACK[0], direction=MAIN_TRACK[1], frame=MAIN_TRACK[2]),
            info=dict(anchors=R["anchors"], margins=R["margins_db"], noise=R["noise_km2"], bays=R["bays_km2"],
                      extent=float(R["extent"].sum() * GEO["px_km2"])), images=images, masks=masks)
json.dump(site, open(f"{OUT}/web/site_data.json", "w"), separators=(",", ":"))
with open(f"{OUT}/web/series.csv", "w") as f:
    cols = list(rows[0].keys())
    f.write(",".join(cols) + "\n")
    for r in rows:
        f.write(",".join(str(r[c]) for c in cols) + "\n")
if os.path.exists("template.html"):
    open(f"{OUT}/web/index.html", "w").write(open("template.html").read().replace("__DATA__", json.dumps(site, separators=(",", ":"))))
    print("index.html written")
zname = "toktogul_out.zip"
with zipfile.ZipFile(zname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(f"{OUT}/web")):
        z.write(f"{OUT}/web/{fn}", f"web/{fn}")
print(f"Saved {zname}: {os.path.getsize(zname) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zname)
except Exception:
    pass
